# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook demonstrates how to load and explore the FAIR² dataset using the `mlcroissant` library. It follows the structure of the provided notebook template and utilizes the Croissant schema at its URL for reproducible, FAIR data access.

### Dataset Source

FAIR² schema URL: [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the FAIR² Croissant URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

metadata = dataset.metadata
print(f"Dataset Name: {metadata.name}\nDescription: {metadata.description}\n\nPublished: {metadata.date_published}")

## 2. Data Overview

We will list all available record sets and their respective fields (columns), using their `@id` for precise referencing. This helps us select entities properly in the next steps.

In [ ]:
# List all record sets and their fields by @id
for record_set in metadata.record_sets or []:
    print(f"RecordSet: {record_set.name} (@id={record_set.id})")
    if record_set.fields:
        for field in record_set.fields:
            column_ids = [c.id for c in field.columns] if field.columns else []
            print(f"  Field: {field.name} (@id={field.id}) | Data type: {field.data_type}")
            for col in field.columns:
                print(f"    Column: {col.name} (@id={col.id})")
    print()

## 3. Data Extraction

We'll extract data from each `RecordSet` defined in the metadata, and load them into pandas DataFrames for analysis. All selections use their Croissant `@id` for referencing.

*(If the printed field/record set names above reference e.g. '7154140-table1' for protein data, use that id in the code below. Adjust accordingly if more or fewer record sets are present.)*

In [ ]:
# Gather all record set IDs
record_sets = [rs.id for rs in metadata.record_sets]
print(f"Record set @id values found: {record_sets}")

dataframes = {}
for record_set_id in record_sets:
    print(f"Loading records for RecordSet {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

# For demonstration, take the first record set for further steps:
if record_sets:
    main_record_set_id = record_sets[0]
    print(f'Columns in {main_record_set_id}: {dataframes[main_record_set_id].columns.tolist()}')
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)

Let's select a numeric field for filtering, normalization, and EDA. We reference all fields by their `@id`. For example, if a column/field like `coverage_percent` or `molecular_weight` exists (check previous output for actual column name/ids), we will use its `@id`.

> ⚠️ **Update the field selection below to match a numeric column @id present in your main record set DataFrame.**

In [ ]:
# Set the main record set you want to analyze
df = dataframes[main_record_set_id]

# Select a numeric field @id available in the dataset
print(f"Available columns in selected DataFrame: {df.columns.tolist()}")
# Example: choose a field like 'molecular_weight' or similar, update accordingly
numeric_field = None
for col in df.columns:
    # Try to pick a float/integer column
    if pd.api.types.is_numeric_dtype(df[col]):
        numeric_field = col
        break

if numeric_field is not None:
    print(f"Analyzing numeric field: {numeric_field}")
    threshold = df[numeric_field].quantile(0.9)  # Use 90th percentile as example
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold:.2f} (top 10%): {len(filtered_df)} rows")
    
    # Normalize
    filtered_df[f"{numeric_field}_normalized"] = (
        filtered_df[numeric_field] - filtered_df[numeric_field].mean()
    ) / filtered_df[numeric_field].std()
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
    
    # Try grouping by a likely categorical field
    group_field = None
    for col in df.columns:
        # Pick first object type column (not current numeric field)
        if col != numeric_field and df[col].dtype == object:
            group_field = col
            break

    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Grouped by '{group_field}', mean of '{numeric_field}':")
        print(grouped_df.head())
else:
    print("No numeric field found for EDA.")

## 5. Visualization

Let's visualize the distribution of the selected numeric field, and if applicable, the group-based means.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of '{numeric_field}'")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()

    if group_field:
        plt.figure(figsize=(10, 4))
        sns.barplot(x=group_field, y=numeric_field, data=grouped_df)
        plt.title(f"Mean {numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(f"Mean {numeric_field}")
        plt.tight_layout()
        plt.show()

## 6. Conclusion

In this notebook, we've demonstrated how to:
- Load FAIR² metadata and records using `mlcroissant`
- Enumerate record sets and fields with their `@id`
- Extract structured tabular data dynamically
- Apply basic EDA tasks like filtering, normalization, grouping, and visualization

This workflow ensures reproducibility and proper referencing for scientific FAIR datasets using Croissant.